In [1]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover

print("Imports successful.")

Imports successful.


In [2]:
# Load the cleaned library from Step 2
df_raw = pd.read_csv("orexin_ligand_library_clean.csv")
df_raw = df_raw.copy()

print(f"Loaded {len(df_raw)} compounds with {len(df_raw.columns)} columns.")
print(f"\nDrug classes present:")
print(df_raw["drug_class"].value_counts())

Loaded 80 compounds with 40 columns.

Drug classes present:
drug_class
DORA_analog       57
agonist_analog    19
DORA               3
agonist            1
Name: count, dtype: int64


In [3]:
# ============================================================
# 3a. Salt removal using RDKit
# ============================================================

remover = SaltRemover()

def strip_salts(smiles):
    """
    Remove salt counterions from a SMILES string.
    Returns the largest organic fragment.
    """
    if pd.isna(smiles):
        return None
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        stripped = remover.StripMol(mol)
        return Chem.MolToSmiles(stripped)
    except:
        return None

df_raw["clean_smiles"] = df_raw["isomeric_smiles"].apply(strip_salts)

# Check how many SMILES were affected
changed = (df_raw["clean_smiles"] != df_raw["isomeric_smiles"]).sum()
failed  = df_raw["clean_smiles"].isna().sum()

print(f"Salt stripping complete.")
print(f"  SMILES modified: {changed}")
print(f"  SMILES failed:   {failed}")
print(f"  Unchanged:       {80 - changed - failed}")

Salt stripping complete.
  SMILES modified: 80
  SMILES failed:   0
  Unchanged:       0


In [4]:
#the former salt stripping appears overly aggressive. Checking to see what happened:

# Compare original vs stripped SMILES for seed compounds
print("Comparing original vs stripped SMILES for seed compounds:")
seeds = df_raw[df_raw["drug_class"].isin(["DORA", "agonist"])].head(4)

for _, row in seeds.iterrows():
    print(f"\n{row['name']}:")
    print(f"  Original: {row['isomeric_smiles'][:60]}...")
    print(f"  Stripped: {row['clean_smiles'][:60]}...")
    print(f"  Same length: {len(str(row['isomeric_smiles'])) == len(str(row['clean_smiles']))}")

Comparing original vs stripped SMILES for seed compounds:

Daridorexant:
  Original: CC1=C(C=CC2=C1N=C(N2)[C@@]3(CCCN3C(=O)C4=C(C=CC(=C4)OC)N5N=C...
  Stripped: COc1ccc(-n2nccn2)c(C(=O)N2CCC[C@@]2(C)c2nc3c(C)c(Cl)ccc3[nH]...
  Same length: False

Lemborexant:
  Original: CC1=NC(=NC=C1OC[C@]2(C[C@H]2C(=O)NC3=NC=C(C=C3)F)C4=CC(=CC=C...
  Stripped: Cc1ncc(OC[C@@]2(c3cccc(F)c3)C[C@H]2C(=O)Nc2ccc(F)cn2)c(C)n1...
  Same length: False

Suvorexant:
  Original: C[C@@H]1CCN(CCN1C(=O)C2=C(C=CC(=C2)C)N3N=CC=N3)C4=NC5=C(O4)C...
  Stripped: Cc1ccc(-n2nccn2)c(C(=O)N2CCN(c3nc4cc(Cl)ccc4o3)CC[C@H]2C)c1...
  Same length: False

Oveporexton:
  Original: CCS(=O)(=O)N[C@@H]1[C@@H](N(CC1(F)F)C(=O)C(C)(C)O)CC2=C(C(=C...
  Stripped: CCS(=O)(=O)N[C@@H]1[C@H](Cc2cccc(-c3cc(F)cc(F)c3)c2F)N(C(=O)...
  Same length: False


In [5]:
# Check the specific salt compound we identified earlier
salt_compound = df_raw[df_raw["cid"] == 91809208]
if len(salt_compound) > 0:
    row = salt_compound.iloc[0]
    print(f"Name: {row['name']}")
    print(f"Original SMILES: {row['isomeric_smiles']}")
    print(f"Stripped SMILES: {row['clean_smiles']}")
else:
    print("CID 91809208 not found in library")

Name: similar_to_91801202_91809208
Original SMILES: CC1=C(C=CC2=C1N=C(N2)[C@@]3(CCCN3C(=O)C4=C(C=CC(=C4)OC)N5N=CC=N5)C)Cl.Cl
Stripped SMILES: COc1ccc(-n2nccn2)c(C(=O)N2CCC[C@@]2(C)c2nc3c(C)c(Cl)ccc3[nH]2)c1


In [6]:
# ============================================================
# Remove duplicates based on clean_smiles
# (catches salt forms that reduce to the same molecule)
# ============================================================

n_before = len(df_raw)
df_salts_removed = df_raw.drop_duplicates(subset="clean_smiles", keep="first")
n_after = len(df_salts_removed)

print(f"Before deduplication: {n_before} compounds")
print(f"After deduplication:  {n_after} compounds")
print(f"Duplicates removed:   {n_before - n_after}")
print(f"\nDrug classes remaining:")
print(df_salts_removed["drug_class"].value_counts())

Before deduplication: 80 compounds
After deduplication:  79 compounds
Duplicates removed:   1

Drug classes remaining:
drug_class
DORA_analog       56
agonist_analog    19
DORA               3
agonist            1
Name: count, dtype: int64


In [7]:
# Save deduplicated library as a checkpoint
df_clean = df_salts_removed.copy()

df_clean.to_csv("orexin_library_step3_input.csv", index=False)
print(f"Saved: orexin_library_step3_input.csv")
print(f"Ready for screening: {len(df_clean)} compounds")

Saved: orexin_library_step3_input.csv
Ready for screening: 79 compounds


In [8]:
# ============================================================
# Stage A — Hard filters (binary pass/fail)
# ============================================================

n_start = len(df_clean)

# Must have a clean SMILES string
df_filtered = df_clean[df_clean["clean_smiles"].notna()].copy()

# Molecular weight: 200-600 Da
df_filtered = df_filtered[df_filtered["molecular_weight"].between(200, 600)]

# LogP: -1 to 6
df_filtered = df_filtered[df_filtered["xlogp"].between(-1, 6)]

# H-bond donors: 0-5
df_filtered = df_filtered[df_filtered["h_bond_donors"].between(0, 5)]

# H-bond acceptors: 0-10
df_filtered = df_filtered[df_filtered["h_bond_acceptors"].between(0, 10)]

# TPSA: under 140 Å²
df_filtered = df_filtered[df_filtered["tpsa"] <= 140]

# No more than 2 Ro5 violations
df_filtered = df_filtered[df_filtered["ro5_violations"] <= 2]

print(f"Before hard filters: {n_start} compounds")
print(f"After hard filters:  {len(df_filtered)} compounds")
print(f"Removed:             {n_start - len(df_filtered)} compounds")
print(f"\nDrug classes remaining:")
print(df_filtered["drug_class"].value_counts())

Before hard filters: 79 compounds
After hard filters:  79 compounds
Removed:             0 compounds

Drug classes remaining:
drug_class
DORA_analog       56
agonist_analog    19
DORA               3
agonist            1
Name: count, dtype: int64


In [9]:
# ============================================================
# Stage B — CNS-specific scoring
# ============================================================

def score_mw(mw):
    if 300 <= mw <= 450:   return 1.0
    if 200 <= mw < 300:    return 0.6
    if 450 < mw <= 550:    return 0.6
    return 0.2

def score_logp(lp):
    if pd.isna(lp):        return np.nan
    if 1.0 <= lp <= 4.0:   return 1.0
    if 0.0 <= lp < 1.0:    return 0.6
    if 4.0 < lp <= 5.0:    return 0.6
    return 0.2

def score_tpsa(t):
    if pd.isna(t):         return np.nan
    if t <= 70:            return 1.0
    if t <= 90:            return 0.7
    if t <= 120:           return 0.4
    return 0.1

def score_hbd(hbd):
    if hbd == 0:           return 1.0
    if hbd == 1:           return 0.8
    if hbd == 2:           return 0.5
    return 0.2

def score_rotbonds(rb):
    if rb <= 5:            return 1.0
    if rb <= 8:            return 0.7
    if rb <= 10:           return 0.4
    return 0.1

df_filtered["score_mw"]       = df_filtered["molecular_weight"].apply(score_mw)
df_filtered["score_logp"]     = df_filtered["xlogp"].apply(score_logp)
df_filtered["score_tpsa"]     = df_filtered["tpsa"].apply(score_tpsa)
df_filtered["score_hbd"]      = df_filtered["h_bond_donors"].apply(score_hbd)
df_filtered["score_rotbonds"] = df_filtered["rotatable_bonds"].apply(score_rotbonds)

# Add ADMET scores
df_filtered["score_bbb"]  = df_filtered["BBB_Martins"].fillna(0.5)
df_filtered["score_herg"] = 1.0 - df_filtered["hERG"].fillna(0.5)
df_filtered["score_ames"] = 1.0 - df_filtered["AMES"].fillna(0.5)

print("CNS scoring complete.")
print(f"\nMean scores by drug class:")
score_cols = ["score_mw", "score_logp", "score_tpsa",
              "score_bbb", "score_herg"]
print(df_filtered.groupby("drug_class")[score_cols].mean().round(3))

CNS scoring complete.

Mean scores by drug class:
                score_mw  score_logp  score_tpsa  score_bbb  score_herg
drug_class                                                             
DORA               0.733       0.733       0.700      0.812       0.168
DORA_analog        0.857       0.857       0.625      0.762       0.182
agonist            0.600       1.000       0.400      0.718       0.115
agonist_analog     0.789       0.916       0.574      0.881       0.165


In [10]:
# ============================================================
# Stage C — Weighted composite score and ranking
# ============================================================

weights = {
    "score_mw":       0.10,
    "score_logp":     0.15,
    "score_tpsa":     0.15,
    "score_hbd":      0.10,
    "score_rotbonds": 0.05,
    "score_bbb":      0.25,
    "score_herg":     0.15,
    "score_ames":     0.05,
}

df_filtered["composite_score"] = df_filtered[list(weights.keys())].apply(
    lambda row: np.nansum(
        [row[c] * w for c, w in weights.items()]
    ) / sum(w for c, w in weights.items() if pd.notna(row[c])),
    axis=1
)

df_filtered = df_filtered.sort_values("composite_score", ascending=False)
df_filtered["rank"] = range(1, len(df_filtered) + 1)
df_filtered["shortlisted"] = df_filtered["rank"] <= 20

print("Top 15 ranked compounds:")
print(df_filtered[["rank", "name", "drug_class",
                    "composite_score", "BBB_Martins",
                    "hERG", "molecular_weight"]].head(15).to_string(index=False))

Top 15 ranked compounds:
 rank                          name  drug_class  composite_score  BBB_Martins     hERG  molecular_weight
    1  similar_to_56944144_67473934 DORA_analog         0.780719     0.913585 0.651404             374.4
    2  similar_to_56944144_56944522 DORA_analog         0.778408     0.870695 0.636354             422.5
    3  similar_to_56944144_56944044 DORA_analog         0.777564     0.917090 0.681296             392.4
    4  similar_to_56944144_67283149 DORA_analog         0.775976     0.926275 0.711141             406.5
    5  similar_to_56944144_56944620 DORA_analog         0.772382     0.878528 0.686526             440.4
    6  similar_to_56944144_67473922 DORA_analog         0.771801     0.910113 0.729783             388.5
    7  similar_to_56944144_56944429 DORA_analog         0.771063     0.901769 0.697812             428.4
    8                   Lemborexant        DORA         0.770513     0.921039 0.730315             410.4
    9  similar_to_56944144_569

In [11]:
# ============================================================
# Save ranked library and shortlist
# ============================================================

df_ranked = df_filtered.copy()

# Save full ranked library
df_ranked.to_csv("orexin_library_ranked.csv", index=False)

# Save top 20 shortlist for docking
shortlist = df_ranked[df_ranked["shortlisted"]].copy()
shortlist.to_csv("orexin_shortlist_for_docking.csv", index=False)

# Save SMILES for docking input
shortlist[["name", "clean_smiles"]].to_csv(
    "shortlist_smiles.csv", index=False
)

print(f"Saved: orexin_library_ranked.csv ({len(df_ranked)} compounds)")
print(f"Saved: orexin_shortlist_for_docking.csv ({len(shortlist)} compounds)")
print(f"Saved: shortlist_smiles.csv (SMILES for docking input)")

print(f"\nShortlist composition:")
print(shortlist["drug_class"].value_counts())

print(f"\nKnown drugs in shortlist:")
known = shortlist[shortlist["drug_class"].isin(["DORA", "agonist"])]
print(known[["rank", "name", "drug_class", "composite_score"]])

Saved: orexin_library_ranked.csv (79 compounds)
Saved: orexin_shortlist_for_docking.csv (20 compounds)
Saved: shortlist_smiles.csv (SMILES for docking input)

Shortlist composition:
drug_class
DORA_analog       18
DORA               1
agonist_analog     1
Name: count, dtype: int64

Known drugs in shortlist:
   rank         name drug_class  composite_score
1     8  Lemborexant       DORA         0.770513
